# Cải thiện Naive Bayes: thí nghiệm trên tập phát triển (train + val)

Notebook này **không đọc nội dung `datasets/splits/test.csv`**. Việc chấm test nằm riêng ở `naive_bayes_final_test.ipynb`.
Thiết kế, lưới tham số và quy tắc chấp nhận được khóa trước trong `experiment_plan.md`.

**Phân bổ phần cứng (đã đo):**
- NB, TF-IDF theo fold, bootstrap: tensor PyTorch trên A100.
- LogReg, LinearSVC, NB-LR: scikit-learn trên CPU, 8 worker, mỗi worker 1 luồng BLAS. Các mô hình này chỉ dùng để so sánh.

Node có 128 lõi nhưng job được 8 lõi, nên phải giới hạn luồng BLAS. Nếu không, một fit LogReg mất 6–8 giây thay vì 0.08 giây.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import hashlib, itertools, json, math, platform, re, subprocess, sys, time
from collections import Counter, namedtuple
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sps
import sklearn
import torch
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda")
DTYPE = torch.float64
N_JOBS = len(os.sched_getaffinity(0))

ROOT = Path.cwd()
while not (ROOT / "Data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SPLITS = ROOT / "models" / "Naive_Bayes" / "datasets" / "splits"
XLSX = ROOT / "Data" / "final_labeled_clean_v3.xlsx"
NBDIR = ROOT / "models" / "Naive_Bayes"
EXP = NBDIR / "experiments"
(EXP / "oof").mkdir(parents=True, exist_ok=True)

LABELS = ["Negative", "Neutral", "Positive"]
L2I = {l: i for i, l in enumerate(LABELS)}
C = len(LABELS)

LOCKED = {
    "models/Naive_Bayes/datasets/splits/train.csv": "c04400673765183a0c904d447b611f48b74389c1b68059d517237d4cdab38505",
    "models/Naive_Bayes/datasets/splits/val.csv": "155c13a615172a7b06c4f2b647e222736d0354dcb603ad90c0c027ecd67f31c7",
    "models/Naive_Bayes/datasets/splits/test.csv": "6135fb08b390b6743968a1fb1dde8319977e3297bad656f00e1d669f972b9ae2",
    "Data/final_labeled_clean_v3.xlsx": "ea6c578763a8396f4c065696e50df0431a0a978cfff40399818e2a7b4ef7a89c",
}
for rel, want in LOCKED.items():
    got = hashlib.sha256((ROOT / rel).read_bytes()).hexdigest()
    assert got == want, f"File bị thay đổi: {rel}"

print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "| sklearn", sklearn.__version__,
      "| numpy", np.__version__, "| workers", N_JOBS, "| OMP", os.environ["OMP_NUM_THREADS"])
print("Hash khớp cho", len(LOCKED), "file bị khóa")

torch 2.8.0+cu128 | NVIDIA A100-PCIE-40GB | sklearn 1.6.1 | numpy 2.0.2 | workers 8 | OMP 1
Hash khớp cho 4 file bị khóa


## 1. Tập phát triển và các fold

Dev = train + val (1800 dòng). Từ xlsx chỉ lấy các dòng thuộc dev, nên `RAW Text` và `Clean Text Old` của test không được nạp.

In [2]:
def read_split(name):
    return pd.read_csv(SPLITS / f"{name}.csv", encoding="utf-8-sig", keep_default_na=False)

dev = pd.concat([read_split("train").assign(origin="train"), read_split("val").assign(origin="val")], ignore_index=True)
xl = pd.read_excel(XLSX, keep_default_na=False, usecols=["Comment ID", "RAW Text", "Clean Text", "Final Label", "Clean Text Old"])
xl = xl[xl["Comment ID"].isin(dev["Comment ID"])].set_index("Comment ID").loc[dev["Comment ID"]].reset_index()
assert (xl["Clean Text"].astype(str).values == dev["Clean Text"].astype(str).values).all()
assert (xl["Final Label"].values == dev["Final Label"].values).all()
dev["Clean Text Old"] = xl["Clean Text Old"].astype(str).values
dev["RAW Text"] = xl["RAW Text"].astype(str).values
del xl

N = len(dev)
y_np = dev["Final Label"].map(L2I).to_numpy()
gid_np = pd.factorize(dev["Clean Text"])[0]
y = torch.tensor(y_np, device=DEVICE)
gid = torch.tensor(gid_np, device=DEVICE)
is_train = torch.tensor((dev["origin"] == "train").to_numpy(), device=DEVICE)
print("dev:", N, "| group:", int(gid.max()) + 1, "|", dev["Final Label"].value_counts().to_dict())

R, K, Q = 5, 5, 3
F_np = np.zeros((R, N), dtype=np.int64)
I_np = np.zeros((R, K, N), dtype=np.int64)
dummy = np.zeros(N)
for r in range(R):
    sgkf = StratifiedGroupKFold(K, shuffle=True, random_state=r)
    for k, (_, te) in enumerate(sgkf.split(dummy, y_np, gid_np)):
        F_np[r, te] = k
    for k in range(K):
        idx = np.where(F_np[r] != k)[0]
        inner = StratifiedGroupKFold(Q, shuffle=True, random_state=100 * r + k)
        for q, (_, te) in enumerate(inner.split(dummy[idx], y_np[idx], gid_np[idx])):
            I_np[r, k, idx[te]] = q

for r in range(R):
    for k in range(K):
        a, b = set(gid_np[F_np[r] == k]), set(gid_np[F_np[r] != k])
        assert not (a & b), "group lẫn qua outer fold"
        idx = np.where(F_np[r] != k)[0]
        for q in range(Q):
            a, b = set(gid_np[idx[I_np[r, k, idx] == q]]), set(gid_np[idx[I_np[r, k, idx] != q]])
            assert not (a & b), "group lẫn qua inner fold"
print("Không có group nào lẫn qua outer/inner fold")

Ft, It = torch.tensor(F_np, device=DEVICE), torch.tensor(I_np, device=DEVICE)
pd.DataFrame({"Comment ID": dev["Comment ID"], **{f"fold_r{r}": F_np[r] for r in range(R)}}).to_csv(EXP / "cv_folds.csv", index=False)

dev: 1800 | group: 1748 | {'Positive': 1091, 'Neutral': 392, 'Negative': 317}


Không có group nào lẫn qua outer/inner fold


## 2. Lõi tensor

Mỗi văn bản được phân tích một lần thành bộ ba `(doc, term, tf)` dạng COO trên GPU. Trong mỗi fold, `df`, vocabulary và IDF được tính lại **chỉ từ `tr`**, nên không cần tách token lại và không rò rỉ thông tin từ phần đánh giá.

Với $x_{dt}$ là trọng số của term $t$ trong văn bản $d$ và $V$ là số term trong vocabulary:

$$\theta_{ct}=\frac{\alpha+\sum_{d:y_d=c}x_{dt}}{\alpha V+\sum_{t'}\sum_{d:y_d=c}x_{dt'}},\qquad
\hat y=\arg\max_c\Big[\ln\pi_c+\sum_t x_{dt}\ln\theta_{ct}\Big]$$

ComplementNB ước lượng $\tilde\theta_{ct}$ từ mọi lớp **trừ** $c$, rồi dự đoán bằng $\arg\max_c\big[-\sum_t x_{dt}\ln\tilde\theta_{ct}\big]$.

Các hàm nhận vector `alphas` có $A$ phần tử và trả tensor $(A,n,C)$, nên nhiều giá trị $\alpha$ được tính song song trong một lần.

In [3]:
TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")          # cùng quy ước với sklearn và notebook gốc

def tokens(text):
    return TOKEN_RE.findall(text.lower())

def word_terms(toks):
    return [(w, 1) for w in toks] + [(f"{toks[i]} {toks[i + 1]}", 2) for i in range(len(toks) - 1)]

def char_wb_terms(text, lo=2, hi=5):
    # cùng thuật toán với analyzer='char_wb' của sklearn; '_' (dấu tách từ ghép) đổi thành khoảng trắng
    out = []
    for w in text.lower().replace("_", " ").split():
        w = f" {w} "
        for n in range(lo, hi + 1):
            off = 0
            out.append((w[off:off + n], n))
            while off + n < len(w):
                off += 1
                out.append((w[off:off + n], n))
            if off == 0:
                break
    return out

class Corpus:
    def __init__(self, term_lists):
        idx, order, d_, t_, f_ = {}, [], [], [], []
        for i, terms in enumerate(term_lists):
            for (w, o), c in Counter(terms).items():
                j = idx.get(w)
                if j is None:
                    j = idx[w] = len(idx)
                    order.append(o)
                d_.append(i); t_.append(j); f_.append(c)
        self.vocab = list(idx)
        self.doc = torch.tensor(d_, device=DEVICE)
        self.term = torch.tensor(t_, device=DEVICE)
        self.tf = torch.tensor(f_, dtype=DTYPE, device=DEVICE)
        self.order = torch.tensor(order, device=DEVICE)
        self.n_docs, self.n_terms = len(term_lists), len(idx)

Feat = namedtuple("Feat", "d t v vm")

def vectorize(cp, tr, min_df=2, max_order=2, weighting="tfidf", keep=None, df_mask=None):
    dfm = tr if df_mask is None else df_mask
    ok = cp.order[cp.term] <= max_order
    df = torch.bincount(cp.term[dfm[cp.doc] & ok], minlength=cp.n_terms).to(DTYPE)
    vm = df >= min_df
    if keep is not None:
        vm = vm & keep
    m = vm[cp.term]
    d, t, f = cp.doc[m], cp.term[m], cp.tf[m]
    if weighting == "tfidf":
        n = dfm.sum().to(DTYPE)
        idf = torch.log((1 + n) / (1 + df)) + 1.0
        v = (1.0 + torch.log(f)) * idf[t]
        norm2 = torch.zeros(cp.n_docs, dtype=DTYPE, device=DEVICE).index_add_(0, d, v * v)
        v = v / norm2.sqrt()[d].clamp_min(1e-12)
    elif weighting == "count":
        v = f
    elif weighting == "binary":
        v = torch.ones_like(f)
    else:
        raise ValueError(weighting)
    return Feat(d, t, v, vm)

def nb_weights(ft, yy, tr, alphas, kind="mnb", w=None):
    k = tr[ft.d]
    vals = ft.v[k] if w is None else ft.v[k] * w[ft.d[k]]
    fc = torch.zeros(C, ft.vm.numel(), dtype=DTYPE, device=DEVICE)
    fc.index_put_((yy[ft.d[k]], ft.t[k]), vals, accumulate=True)
    if kind == "cnb":
        fc = fc.sum(0, keepdim=True) - fc
    s = (fc[None] + alphas.view(-1, 1, 1)) * ft.vm
    lt = torch.log(s.clamp_min(1e-300)) - torch.log(s.sum(2, keepdim=True))
    return -lt if kind == "cnb" else lt

def nb_scores(ft, lt, ev):
    n_ev = int(ev.sum())
    pos = torch.full((ev.numel(),), -1, device=DEVICE)
    pos[ev] = torch.arange(n_ev, device=DEVICE)
    e = ev[ft.d]
    jl = torch.zeros(lt.shape[0], n_ev, C, dtype=DTYPE, device=DEVICE)
    jl.index_add_(1, pos[ft.d[e]], ft.v[e].view(1, -1, 1) * lt[:, :, ft.t[e]].transpose(1, 2))
    return jl

def log_prior(yy, tr, w=None):
    cc = torch.zeros(C, dtype=DTYPE, device=DEVICE)
    cc.index_add_(0, yy[tr], torch.ones(int(tr.sum()), dtype=DTYPE, device=DEVICE) if w is None else w[tr])
    return torch.log(cc / cc.sum())

def select_terms(ft, yy, tr, topk, method):
    # bảng 3×2 (lớp × có/không có term), tính chỉ trên văn bản train
    k = tr[ft.d]
    O = torch.zeros(C, ft.vm.numel(), dtype=DTYPE, device=DEVICE)
    O.index_put_((yy[ft.d[k]], ft.t[k]), torch.ones(int(k.sum()), dtype=DTYPE, device=DEVICE), accumulate=True)
    Nc = torch.bincount(yy[tr], minlength=C).to(DTYPE).view(C, 1)
    n = Nc.sum()
    A, B = O, Nc - O
    n1 = A.sum(0, keepdim=True)
    E1, E0 = Nc * n1 / n, Nc * (n - n1) / n
    if method == "chi2":
        s = ((A - E1) ** 2 / E1.clamp_min(1e-12) + (B - E0) ** 2 / E0.clamp_min(1e-12)).sum(0)
    else:
        xl = lambda a, e: torch.where(a > 0, (a / n) * torch.log(a / e.clamp_min(1e-12)), torch.zeros_like(a))
        s = (xl(A, E1) + xl(B, E0)).sum(0)
    s = s.masked_fill(~ft.vm, -1.0)
    keep = torch.zeros_like(ft.vm)
    keep[s.topk(min(topk, int(ft.vm.sum()))).indices] = True
    return keep

def macro_f1_cm(cm):
    tp = cm.diagonal(dim1=-2, dim2=-1)
    prec = tp / cm.sum(-2).clamp_min(1e-12)
    rec = tp / cm.sum(-1).clamp_min(1e-12)
    return 2 * prec * rec / (prec + rec).clamp_min(1e-12)

def cm_batch(yy, pred):                            # pred: (K, n) -> (K, C, C); hàng = nhãn thật
    code = yy.view(1, -1) * C + pred
    return (torch.zeros(pred.shape[0], C * C, dtype=DTYPE, device=DEVICE)
            .scatter_add_(1, code, torch.ones_like(code, dtype=DTYPE)).view(-1, C, C))

def score_macro(cm):
    return macro_f1_cm(cm).mean(-1)

CORPORA = {}
def corpus(key, build):
    if key not in CORPORA:
        CORPORA[key] = Corpus(build())
    return CORPORA[key]

T_CLEAN = [tokens(t) for t in dev["Clean Text"]]
corpus("clean", lambda: [word_terms(t) for t in T_CLEAN])
print("clean: n_terms", CORPORA["clean"].n_terms, "| nnz", CORPORA["clean"].doc.numel())

clean: n_terms 25891 | nnz 59501


### Kiểm tra đúng đắn

So sánh với `scikit-learn` và một bản tham chiếu naive trên dữ liệu thật:
TF-IDF, MNB, ComplementNB, macro-F1, bootstrap theo group, và tái lập kết quả val đã công bố (0.6147) khi fit trên train, chấm trên val.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.naive_bayes import ComplementNB, MultinomialNB

cp = CORPORA["clean"]
tr_m, va_m = is_train, ~is_train
tr_txt = dev.loc[dev.origin == "train", "Clean Text"].tolist()
va_txt = dev.loc[dev.origin == "val", "Clean Text"].tolist()

ft = vectorize(cp, tr_m, 2, 2, "tfidf")
sk = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(tr_txt)
assert int(ft.vm.sum()) == len(sk.get_feature_names_out()) == 4204
vocab_np = np.array(cp.vocab)
colmap = {w: i for i, w in enumerate(sk.get_feature_names_out())}

def to_dense(ft_, mask):
    idx = torch.full((mask.numel(),), -1, device=DEVICE); idx[mask] = torch.arange(int(mask.sum()), device=DEVICE)
    e = mask[ft_.d]
    cols = np.array([colmap[w] for w in vocab_np[ft_.t[e].cpu().numpy()]])
    out = np.zeros((int(mask.sum()), len(colmap)))
    out[idx[ft_.d[e]].cpu().numpy(), cols] = ft_.v[e].cpu().numpy()
    return out

Xtr, Xva = to_dense(ft, tr_m), to_dense(ft, va_m)
print("TF-IDF max|diff| train:", np.abs(Xtr - sk.transform(tr_txt).toarray()).max(), "| val:", np.abs(Xva - sk.transform(va_txt).toarray()).max())

alphas = torch.tensor([0.01], dtype=DTYPE, device=DEVICE)
jl = nb_scores(ft, nb_weights(ft, y, tr_m, alphas), va_m)[0]
sk_nb = MultinomialNB(alpha=0.01, fit_prior=False).fit(Xtr, y_np[is_train.cpu().numpy()])
p = jl.argmax(1).cpu().numpy()
agree = (p == sk_nb.predict(Xva)).mean()
f1 = float(score_macro(cm_batch(y[va_m], jl.argmax(1)[None]))[0])
print("MNB trùng sklearn:", agree, "| macro-F1 val:", f1, "| sklearn:", f1_score(y_np[~is_train.cpu().numpy()], p, average="macro"))
assert agree == 1.0 and abs(f1 - 0.6146723400992851) < 1e-12, "không tái lập được kết quả val đã công bố"

jc = nb_scores(ft, nb_weights(ft, y, tr_m, alphas, "cnb"), va_m)[0]
sk_cnb = ComplementNB(alpha=0.01).fit(Xtr, y_np[is_train.cpu().numpy()])
print("CNB trùng sklearn:", (jc.argmax(1).cpu().numpy() == sk_cnb.predict(Xva)).mean(),
      "| max|diff| jll:", np.abs(jc.cpu().numpy() - sk_cnb.predict_joint_log_proba(Xva) ).max())

# bootstrap: trọng số theo group, đối chiếu naive
g = torch.Generator(device=DEVICE).manual_seed(SEED)
B_BOOT, G = 5000, int(gid.max()) + 1
draw = torch.randint(0, G, (B_BOOT, G), generator=g, device=DEVICE)
W_G = torch.zeros(B_BOOT, G, dtype=DTYPE, device=DEVICE).scatter_add_(1, draw, torch.ones_like(draw, dtype=DTYPE))
W_BOOT = W_G[:, gid]
del draw, W_G

def boot_scores(pred, yy=y, W=None):               # pred: (R, N) -> (B,)
    W = W_BOOT if W is None else W
    out = 0.0
    for r in range(pred.shape[0]):
        oh = torch.nn.functional.one_hot(yy * C + pred[r], C * C).to(DTYPE)
        out = out + score_macro((W @ oh).view(-1, C, C))
    return out / pred.shape[0]

rng = np.random.default_rng(0)
fake = torch.tensor(rng.integers(0, C, size=(1, N)), device=DEVICE)
fast = boot_scores(fake)
for b in [0, 1, 2]:
    w = W_BOOT[b].cpu().numpy().astype(int)
    naive = f1_score(np.repeat(y_np, w), np.repeat(fake[0].cpu().numpy(), w), average="macro")
    assert abs(naive - float(fast[b])) < 1e-12
for b in range(50):
    w = W_BOOT[b]
    assert all(w[gid == q].unique().numel() == 1 for q in gid.unique()[:30])
print("Bootstrap theo group khớp bản naive và trọng số đồng nhất trong group")

# chọn đặc trưng: chi2 trên bảng 3×2 khớp scipy
from scipy.stats import chi2_contingency
ft0 = vectorize(cp, tr_m, 2, 2, "tfidf")
keep = select_terms(ft0, y, tr_m, 50, "chi2")
some = keep.nonzero().ravel()[0].item()
docs = ft0.d[(ft0.t == some) & tr_m[ft0.d]]
A = torch.bincount(y[docs], minlength=C).cpu().numpy()
Nc = np.bincount(y_np[is_train.cpu().numpy()], minlength=C)
print("chi2 scipy cho một term:", round(chi2_contingency(np.vstack([A, Nc - A]), correction=False)[0], 6))
print("Tất cả kiểm tra đúng đắn đã qua")

TF-IDF max|diff| train: 2.220446049250313e-16 | val: 1.6653345369377348e-16
MNB trùng sklearn: 1.0 | macro-F1 val: 0.6146723400992851 | sklearn: 0.6146723400992852
CNB trùng sklearn: 1.0 | max|diff| jll: 8.526512829121202e-14


Bootstrap theo group khớp bản naive và trọng số đồng nhất trong group
chi2 scipy cho một term: 19.327931
Tất cả kiểm tra đúng đắn đã qua


## 3. Khung nested CV

- `NBVariant`: một họ NB với lưới tham số, chấm được hàng loạt $\alpha$, prior và offset trong một lần.
- `nested_cv`: chọn cấu hình bằng inner CV gộp, fit lại trên outer-train, dự đoán outer-test.
- `compare`: paired bootstrap theo group, dùng chung ma trận trọng số `W_BOOT` cho mọi cặp.
- Mỗi biến thể ghi dự đoán từng mẫu vào `experiments/oof/`.

In [5]:
ALPHAS = (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0)

class NBVariant:
    def __init__(self, name, corpora=("clean",), kind="mnb", weightings=("tfidf",), max_orders=(1, 2), min_dfs=(1, 2, 3),
                 alphas=ALPHAS, priors=(False, True), offsets=((0.0, 0.0),), topks=(None,), weight_fn=None, ext_src=None):
        self.name, self.kind, self.weight_fn = name, kind, weight_fn
        self.alphas = alphas
        self.alpha_t = torch.tensor(alphas, dtype=DTYPE, device=DEVICE)
        self.priors, self.offsets = priors, offsets
        self.off_t = [torch.tensor([a, b, 0.0], dtype=DTYPE, device=DEVICE) for a, b in offsets]
        self.blocks = list(itertools.product(corpora, weightings, max_orders, min_dfs, topks))
        self.inner = list(itertools.product(priors, range(len(offsets)), range(len(alphas))))
        self.configs = [dict(corpus=b[0], weighting=b[1], max_order=b[2], min_df=b[3],
                             topk=None if b[4] is None else f"{b[4][1]}:{b[4][0]}", fit_prior=p,
                             offset=str(offsets[o]), alpha=alphas[a], kind=kind)
                        for b in self.blocks for p, o, a in self.inner]
        self.ext_src = ext_src

    def _masks(self, tr, ev):
        if self.ext_src is None:
            return tr, ev, None, y
        s = self.ext_src
        return (torch.cat([tr, tr[s]]), torch.cat([ev, torch.zeros_like(s, dtype=torch.bool)]),
                torch.cat([tr, torch.zeros_like(s, dtype=torch.bool)]), torch.cat([y, y[s]]))

    def _jll(self, block, tr, ev):
        ck, wt, mo, md, tk = block
        tr2, ev2, dfm, yy = self._masks(tr, ev)
        cp = CORPORA[ck]
        keep = None
        if tk is not None:
            keep = select_terms(vectorize(cp, tr2, md, mo, wt, df_mask=dfm), yy, tr2, tk[0], tk[1])
        ft = vectorize(cp, tr2, md, mo, wt, keep=keep, df_mask=dfm)
        w = None if self.weight_fn is None else self.weight_fn(tr2, yy)
        jl = nb_scores(ft, nb_weights(ft, yy, tr2, self.alpha_t, self.kind, w), ev2)
        lp = log_prior(yy, tr2, w).view(1, 1, C)
        out = []
        for p in self.priors:
            base = jl + lp if p else jl
            for o in self.off_t:
                out.append(base + o.view(1, 1, C))
        return torch.cat(out, 0)                    # (len(inner), n_ev, C)

    def score(self, tr, ev):
        yev = y[ev]
        return torch.cat([cm_batch(yev, self._jll(b, tr, ev).argmax(2)) for b in self.blocks], 0)

    def predict(self, cfg, tr, ev):
        b, i = divmod(cfg, len(self.inner))
        return self._jll(self.blocks[b], tr, ev)[i]

def first_best(f1):
    return int((f1 >= f1.max() - 1e-12).nonzero()[0])

RESULTS, SELECTIONS = {}, []

def nested_cv(v, train_filter=None):
    t0 = time.perf_counter()
    pred = torch.zeros(R, N, dtype=torch.long, device=DEVICE)
    prob = torch.zeros(R, N, C, dtype=DTYPE, device=DEVICE)
    single = len(v.configs) == 1
    for r in range(R):
        for k in range(K):
            otr, ote = Ft[r] != k, Ft[r] == k
            keep = train_filter(r, k, otr) if train_filter else torch.ones_like(otr)
            if single:
                best, inner_f1 = 0, float("nan")
            else:
                cm = 0.0
                for q in range(Q):
                    iv = otr & (It[r, k] == q)
                    cm = cm + v.score(otr & ~iv & keep, iv)
                f = score_macro(cm)
                best, inner_f1 = first_best(f), float(f.max())
            jl = v.predict(best, otr & keep, ote)
            pred[r, ote] = jl.argmax(1)
            prob[r, ote] = torch.softmax(jl, 1)
            SELECTIONS.append(dict(variant=v.name, repeat=r, fold=k, cfg=best, inner_macro_f1=inner_f1, **v.configs[best]))
    return pred, prob, time.perf_counter() - t0

def register(name, pred, prob=None, runtime=float("nan"), note=""):
    per = []
    for r in range(R):
        cm = cm_batch(y, pred[r][None])
        per.append((float(score_macro(cm)), *macro_f1_cm(cm)[0].tolist(), float((pred[r] == y).double().mean())))
    per = np.array(per)
    RESULTS[name] = dict(pred=pred, prob=prob)
    row = dict(variant=name, macro_f1_mean=per[:, 0].mean(), macro_f1_std=per[:, 0].std(ddof=1),
               f1_negative=per[:, 1].mean(), f1_neutral=per[:, 2].mean(), f1_positive=per[:, 3].mean(),
               accuracy=per[:, 4].mean(), n_repeats=R, n_folds=R * K, runtime_s=runtime, note=note)
    b = boot_scores(pred).cpu().numpy()
    row["macro_f1_ci95_lo"], row["macro_f1_ci95_hi"] = np.percentile(b, [2.5, 97.5])
    out = pd.DataFrame({"Comment ID": dev["Comment ID"], "y_true": dev["Final Label"]})
    for r in range(R):
        out[f"pred_r{r}"] = [LABELS[i] for i in pred[r].tolist()]
    out.to_csv(EXP / "oof" / f"{name}.csv.gz", index=False)
    return row

TABLE, COMPARE = [], []

def run(v, note="", train_filter=None):
    pred, prob, t = nested_cv(v, train_filter)
    row = register(v.name, pred, prob, t, note)
    TABLE.append(row)
    print(f"{v.name:14s} macro-F1 {row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f} | "
          f"Neg {row['f1_negative']:.3f} Neu {row['f1_neutral']:.3f} Pos {row['f1_positive']:.3f} | acc {row['accuracy']:.3f} | "
          f"{len(v.configs)} cấu hình, {t:.1f}s")
    return row

def compare(a, b, exp=""):
    pa, pb = RESULTS[a]["pred"], RESULTS[b]["pred"]
    fa = float(np.mean([float(score_macro(cm_batch(y, pa[r][None]))) for r in range(R)]))
    fb = float(np.mean([float(score_macro(cm_batch(y, pb[r][None]))) for r in range(R)]))
    d = (boot_scores(pa) - boot_scores(pb)).cpu().numpy()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    p = max(p, 1 / len(d))
    row = dict(experiment=exp, variant=a, reference=b, f1_variant=fa, f1_reference=fb, delta=fa - fb,
               ci95_lo=lo, ci95_hi=hi, p_boot=p, accepted=bool(lo > 0))
    COMPARE.append(row)
    print(f"{exp:5s} {a} vs {b}: Δ = {fa - fb:+.4f} CI95 [{lo:+.4f}, {hi:+.4f}] p={p:.4f} -> {'CHẤP NHẬN' if lo > 0 else 'không chấp nhận'}")
    return row

def save_tables():
    pd.DataFrame(TABLE).to_csv(EXP / "results_cv.csv", index=False)
    c = pd.DataFrame(COMPARE)
    if len(c):
        o = c["p_boot"].argsort().values
        adj, cur = np.zeros(len(c)), 0.0
        for i, j in enumerate(o):
            cur = max(cur, min(1.0, (len(c) - i) * c["p_boot"].values[j]))
            adj[j] = cur
        c["p_holm"] = adj
        c.to_csv(EXP / "comparisons_cv.csv", index=False)
    pd.DataFrame(SELECTIONS).to_csv(EXP / "selected_configs.csv", index=False)

## 4. E0: đối chứng và tham chiếu

R0 là cấu hình hiện tại, cố định, không tune. Các mô hình sklearn được fit lại trên **cùng fold** để so sánh ghép cặp được. Script sinh `comparison_val_test.csv` không có trong repo, nên không dùng lại số trong đó.

In [6]:
R0 = NBVariant("R0", max_orders=(2,), min_dfs=(2,), alphas=(0.01,), priors=(False,))
run(R0, "cố định, không tune")

maj = torch.full((R, N), 0, dtype=torch.long, device=DEVICE)
for r in range(R):
    for k in range(K):
        otr, ote = Ft[r] != k, Ft[r] == k
        maj[r, ote] = torch.bincount(y[otr], minlength=C).argmax()
TABLE.append(register("Majority", maj, None, 0.0, "luôn đoán lớp đa số của outer-train"))
print("Majority macro-F1:", round(TABLE[-1]["macro_f1_mean"], 4))

R0             macro-F1 0.5928 ± 0.0046 | Neg 0.516 Neu 0.470 Pos 0.792 | acc 0.677 | 1 cấu hình, 0.1s
Majority macro-F1: 0.2516


In [7]:
def csr_from(ft, n_docs):
    cols = torch.full((ft.vm.numel(),), -1, device=DEVICE)
    cols[ft.vm] = torch.arange(int(ft.vm.sum()), device=DEVICE)
    return sps.csr_matrix((ft.v.cpu().numpy(), (ft.d.cpu().numpy(), cols[ft.t].cpu().numpy())),
                          shape=(n_docs, int(ft.vm.sum())))

LR_C = (0.1, 0.3, 1, 3, 10, 30)
SVC_C = (0.03, 0.1, 0.3, 1, 3, 10)

def make_model(kind, c, cw):
    if kind == "logreg":
        return LogisticRegression(C=c, class_weight=cw, max_iter=2000)
    return LinearSVC(C=c, class_weight=cw, max_iter=20000, random_state=0)

def sk_grid_task(Xtr, ytr, Xev, yev, kind, grid):
    out = []
    for c, cw in grid:
        p = make_model(kind, c, cw).fit(Xtr, ytr).predict(Xev)
        out.append(np.bincount(yev * C + p, minlength=C * C))
    return np.array(out)

def sk_final_task(Xtr, ytr, Xev, kind, c, cw):
    m = make_model(kind, c, cw).fit(Xtr, ytr)
    s = m.predict_proba(Xev) if kind == "logreg" else m.decision_function(Xev)
    return m.predict(Xev), s

def sk_nested(name, kind, Cs, corpus_key="clean", max_orders=(1, 2), min_dfs=(1, 2, 3), cws=(None, "balanced"), note=""):
    t0 = time.perf_counter()
    cp = CORPORA[corpus_key]
    blocks = list(itertools.product(max_orders, min_dfs))
    grid = list(itertools.product(Cs, cws))
    tasks, keys = [], []
    for r in range(R):
        for k in range(K):
            otr = Ft[r] != k
            for q in range(Q):
                iv = otr & (It[r, k] == q)
                it = otr & ~iv
                for bi, (mo, md) in enumerate(blocks):
                    X = csr_from(vectorize(cp, it, md, mo, "tfidf"), N)
                    a, b = it.cpu().numpy(), iv.cpu().numpy()
                    tasks.append(delayed(sk_grid_task)(X[a], y_np[a], X[b], y_np[b], kind, grid))
                    keys.append((r, k, bi))
    res = Parallel(n_jobs=N_JOBS)(tasks)
    agg = {}
    for key, cm in zip(keys, res):
        agg[key] = agg.get(key, 0) + cm
    finals, meta = [], []
    for r in range(R):
        for k in range(K):
            f = torch.cat([score_macro(torch.tensor(agg[(r, k, bi)], dtype=DTYPE, device=DEVICE).view(-1, C, C)) for bi in range(len(blocks))])
            best = first_best(f)
            bi, gi = divmod(best, len(grid))
            mo, md = blocks[bi]
            c, cw = grid[gi]
            otr = Ft[r] != k
            X = csr_from(vectorize(cp, otr, md, mo, "tfidf"), N)
            a = otr.cpu().numpy()
            finals.append(delayed(sk_final_task)(X[a], y_np[a], X[~a], kind, c, cw))
            meta.append((r, k, ~a))
            SELECTIONS.append(dict(variant=name, repeat=r, fold=k, cfg=best, inner_macro_f1=float(f.max()), corpus=corpus_key,
                                   max_order=mo, min_df=md, C=c, class_weight=str(cw), kind=kind))
    out = Parallel(n_jobs=N_JOBS)(finals)
    pred = torch.zeros(R, N, dtype=torch.long, device=DEVICE)
    prob = torch.zeros(R, N, C, dtype=DTYPE, device=DEVICE)
    for (r, k, mask), (p, s) in zip(meta, out):
        m = torch.tensor(mask, device=DEVICE)
        pred[r, m] = torch.tensor(p, device=DEVICE)
        prob[r, m] = torch.tensor(s, dtype=DTYPE, device=DEVICE)
    t = time.perf_counter() - t0
    row = register(name, pred, prob, t, note)
    TABLE.append(row)
    print(f"{name:14s} macro-F1 {row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f} | Neg {row['f1_negative']:.3f} "
          f"Neu {row['f1_neutral']:.3f} Pos {row['f1_positive']:.3f} | acc {row['accuracy']:.3f} | {t:.1f}s")
    return row

sk_nested("LogReg_C3", "logreg", (3,), max_orders=(2,), min_dfs=(2,), cws=(None,), note="sklearn, cố định C=3, không class_weight")
sk_nested("SVC_C1", "svc", (1,), max_orders=(2,), min_dfs=(2,), cws=(None,), note="sklearn, cố định C=1, không class_weight")
sk_nested("LogReg_tuned", "logreg", LR_C, note="sklearn, nested")
sk_nested("SVC_tuned", "svc", SVC_C, note="sklearn, nested")
save_tables()

LogReg_C3      macro-F1 0.5690 ± 0.0037 | Neg 0.457 Neu 0.448 Pos 0.802 | acc 0.692 | 2.8s


SVC_C1         macro-F1 0.5826 ± 0.0049 | Neg 0.491 Neu 0.453 Pos 0.804 | acc 0.690 | 0.5s


LogReg_tuned   macro-F1 0.6193 ± 0.0053 | Neg 0.544 Neu 0.525 Pos 0.790 | acc 0.682 | 32.8s


SVC_tuned      macro-F1 0.6008 ± 0.0091 | Neg 0.517 Neu 0.484 Pos 0.801 | acc 0.688 | 9.4s


## 5. E1: tune lại bằng nested CV

R1 tune trên lưới G (84 cấu hình) trong từng outer fold. Đây cũng là đối chứng cho E2–E7, để mọi biến thể được tune như nhau.

In [8]:
R1 = NBVariant("R1")
assert len(R1.configs) == 84
run(R1, "nested trên lưới G")
compare("R1", "R0", "E1")
compare("LogReg_tuned", "R0", "ref")
compare("LogReg_tuned", "R1", "ref")
compare("SVC_tuned", "R1", "ref")
compare("R0", "Majority", "ref")
sel = pd.DataFrame([s for s in SELECTIONS if s["variant"] == "R1"])
print(sel[["alpha", "fit_prior", "min_df", "max_order"]].value_counts().head(10))
save_tables()

R1             macro-F1 0.6128 ± 0.0065 | Neg 0.552 Neu 0.494 Pos 0.792 | acc 0.683 | 84 cấu hình, 0.9s
E1    R1 vs R0: Δ = +0.0200 CI95 [+0.0079, +0.0327] p=0.0016 -> CHẤP NHẬN
ref   LogReg_tuned vs R0: Δ = +0.0265 CI95 [+0.0087, +0.0451] p=0.0048 -> CHẤP NHẬN
ref   LogReg_tuned vs R1: Δ = +0.0065 CI95 [-0.0071, +0.0203] p=0.3400 -> không chấp nhận
ref   SVC_tuned vs R1: Δ = -0.0120 CI95 [-0.0265, +0.0024] p=0.1064 -> không chấp nhận
ref   R0 vs Majority: Δ = +0.3412 CI95 [+0.3169, +0.3645] p=0.0002 -> CHẤP NHẬN
alpha  fit_prior  min_df  max_order
0.10   False      2       2            10
0.30   False      3       2             7
                  2       2             4
0.10   False      3       2             2
0.03   False      2       2             1
                  1       2             1
Name: count, dtype: int64


## 6. E2: trọng số đặc trưng

In [9]:
run(NBVariant("E2a_count", weightings=("count",)), "tf thô")
run(NBVariant("E2b_binary", weightings=("binary",)), "có/không")
compare("E2a_count", "R1", "E2a")
compare("E2b_binary", "R1", "E2b")
save_tables()

E2a_count      macro-F1 0.6039 ± 0.0108 | Neg 0.534 Neu 0.489 Pos 0.789 | acc 0.679 | 84 cấu hình, 0.8s


E2b_binary     macro-F1 0.6073 ± 0.0094 | Neg 0.542 Neu 0.490 Pos 0.790 | acc 0.679 | 84 cấu hình, 0.8s
E2a   E2a_count vs R1: Δ = -0.0089 CI95 [-0.0211, +0.0028] p=0.1368 -> không chấp nhận
E2b   E2b_binary vs R1: Δ = -0.0055 CI95 [-0.0165, +0.0052] p=0.3144 -> không chấp nhận


## 7. E3: biến thể Naive Bayes

**ComplementNB** chạy trên GPU, không dùng prior. **NB-LR** là mô hình lai: với mỗi lớp $c$ theo kiểu one-vs-rest,

$$r_c=\ln\frac{\mathbf p/\lVert\mathbf p\rVert_1}{\mathbf q/\lVert\mathbf q\rVert_1},\quad
\mathbf p=\alpha+\sum_{d:y_d=c}\mathbf f_d,\quad \mathbf q=\alpha+\sum_{d:y_d\ne c}\mathbf f_d,$$

với $\mathbf f_d$ là vector binary. Đặc trưng $\mathbf f_d\circ r_c$ đi vào LogReg nhị phân (scikit-learn, CPU). Dự đoán là lớp có decision score lớn nhất.

In [10]:
run(NBVariant("E3a_cnb", kind="cnb", priors=(False,)), "ComplementNB; prior không dùng khi dự đoán")
compare("E3a_cnb", "R1", "E3a")

def nblr_task(Xtr, ytr, Xev, yev, grid, alpha=1.0, full=False):
    scores = np.zeros((len(grid), Xev.shape[0], C))
    for c in range(C):
        pos = ytr == c
        p = alpha + np.asarray(Xtr[pos].sum(0)).ravel()
        q = alpha + np.asarray(Xtr[~pos].sum(0)).ravel()
        rr = np.log(p / p.sum()) - np.log(q / q.sum())
        Xa, Xb = Xtr.multiply(rr).tocsr(), Xev.multiply(rr).tocsr()
        for i, (cc, cw) in enumerate(grid):
            scores[i, :, c] = LogisticRegression(C=cc, class_weight=cw, max_iter=2000).fit(Xa, pos.astype(int)).decision_function(Xb)
    if full:
        return scores[0]
    return np.array([np.bincount(yev * C + s.argmax(1), minlength=C * C) for s in scores])

def nblr_nested(name, Cs=(0.03, 0.1, 0.3, 1, 3, 10), cws=(None,), min_dfs=(1, 2), cp_key="clean", note=""):
    t0 = time.perf_counter()
    cp = CORPORA[cp_key]
    grid = list(itertools.product(Cs, cws))
    tasks, keys = [], []
    for r in range(R):
        for k in range(K):
            otr = Ft[r] != k
            for q in range(Q):
                iv = otr & (It[r, k] == q)
                it = otr & ~iv
                for mi, md in enumerate(min_dfs):
                    X = csr_from(vectorize(cp, it, md, 2, "binary"), N)
                    a, b = it.cpu().numpy(), iv.cpu().numpy()
                    tasks.append(delayed(nblr_task)(X[a], y_np[a], X[b], y_np[b], grid))
                    keys.append((r, k, mi))
    res = Parallel(n_jobs=N_JOBS)(tasks)
    agg = {}
    for key, cm in zip(keys, res):
        agg[key] = agg.get(key, 0) + cm
    finals, meta = [], []
    for r in range(R):
        for k in range(K):
            f = torch.cat([score_macro(torch.tensor(agg[(r, k, mi)], dtype=DTYPE, device=DEVICE).view(-1, C, C)) for mi in range(len(min_dfs))])
            best = first_best(f)
            mi, gi = divmod(best, len(grid))
            otr = Ft[r] != k
            X = csr_from(vectorize(cp, otr, min_dfs[mi], 2, "binary"), N)
            a = otr.cpu().numpy()
            finals.append(delayed(nblr_task)(X[a], y_np[a], X[~a], y_np[~a], [grid[gi]], 1.0, True))
            meta.append((r, ~a))
            SELECTIONS.append(dict(variant=name, repeat=r, fold=k, cfg=best, inner_macro_f1=float(f.max()), min_df=min_dfs[mi],
                                   C=grid[gi][0], class_weight=str(grid[gi][1]), kind="nblr"))
    out = Parallel(n_jobs=N_JOBS)(finals)
    pred = torch.zeros(R, N, dtype=torch.long, device=DEVICE)
    prob = torch.zeros(R, N, C, dtype=DTYPE, device=DEVICE)
    for (r, mask), s in zip(meta, out):
        m = torch.tensor(mask, device=DEVICE)
        st = torch.tensor(s, dtype=DTYPE, device=DEVICE)
        pred[r, m], prob[r, m] = st.argmax(1), torch.softmax(st, 1)
    t = time.perf_counter() - t0
    row = register(name, pred, prob, t, note or "mô hình lai; LogReg từ sklearn, CPU")
    TABLE.append(row)
    print(f"{name:14s} macro-F1 {row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f} | Neg {row['f1_negative']:.3f} "
          f"Neu {row['f1_neutral']:.3f} Pos {row['f1_positive']:.3f} | acc {row['accuracy']:.3f} | {t:.1f}s")

nblr_nested("E3b_nblr")
compare("E3b_nblr", "R1", "E3b")
# POST-HOC (ngoài bản đăng ký, xem Nhật ký sai lệch): E3b không có class_weight trong lưới, trong khi LogReg_tuned có
nblr_nested("E3b_cw", cws=(None, "balanced"), note="POST-HOC: thêm class_weight vào lưới; mô hình lai; sklearn, CPU")
compare("E3b_cw", "R1", "E3b*")
save_tables()

E3a_cnb        macro-F1 0.5991 ± 0.0077 | Neg 0.540 Neu 0.462 Pos 0.795 | acc 0.680 | 42 cấu hình, 0.9s
E3a   E3a_cnb vs R1: Δ = -0.0137 CI95 [-0.0235, -0.0044] p=0.0032 -> không chấp nhận


E3b_nblr       macro-F1 0.5640 ± 0.0114 | Neg 0.474 Neu 0.420 Pos 0.799 | acc 0.683 | 5.4s
E3b   E3b_nblr vs R1: Δ = -0.0488 CI95 [-0.0669, -0.0309] p=0.0002 -> không chấp nhận


E3b_cw         macro-F1 0.5966 ± 0.0077 | Neg 0.513 Neu 0.494 Pos 0.782 | acc 0.671 | 10.2s
E3b*  E3b_cw vs R1: Δ = -0.0162 CI95 [-0.0321, -0.0004] p=0.0464 -> không chấp nhận


## 8. E4: đầu vào văn bản

- **E4a:** thay `Clean Text` bằng `Clean Text Old`. Hai cột khác nhau ở nhiều điểm (Old giữ `nhưng`, chữ số, token emoji nhưng có ít `không` hơn), nên đây là so sánh trọn gói.
- **E4b:** giữ `Clean Text`, thêm 4 token meta suy ra từ `RAW Text`. Nguyên văn RAW có tên thật nên không được đưa vào đặc trưng.

In [11]:
HI = re.compile(r"\b10\s*(điểm|đ\b|/\s*10)")
LO = re.compile(r"\b[0-5](\s*điểm|\s*/\s*10)")

def meta_terms(raw):
    low, out = raw.lower(), []
    if "?" in raw: out.append(("__qmark__", 1))
    if "!" in raw: out.append(("__excl__", 1))
    if HI.search(low): out.append(("__score_hi__", 1))
    if LO.search(low): out.append(("__score_lo__", 1))
    return out

corpus("old", lambda: [word_terms(tokens(t)) for t in dev["Clean Text Old"]])
corpus("meta", lambda: [word_terms(t) + meta_terms(r) for t, r in zip(T_CLEAN, dev["RAW Text"])])
print("n_terms: old", CORPORA["old"].n_terms, "| meta", CORPORA["meta"].n_terms)
run(NBVariant("E4a_old", corpora=("old",)), "đầu vào Clean Text Old")
run(NBVariant("E4b_meta", corpora=("meta",)), "Clean Text + token meta từ RAW")
compare("E4a_old", "R1", "E4a")
compare("E4b_meta", "R1", "E4b")
save_tables()

n_terms: old 27043 | meta 25895


E4a_old        macro-F1 0.6201 ± 0.0059 | Neg 0.547 Neu 0.517 Pos 0.796 | acc 0.693 | 84 cấu hình, 1.0s


E4b_meta       macro-F1 0.6130 ± 0.0063 | Neg 0.553 Neu 0.491 Pos 0.796 | acc 0.686 | 84 cấu hình, 0.9s
E4a   E4a_old vs R1: Δ = +0.0073 CI95 [-0.0075, +0.0219] p=0.3272 -> không chấp nhận
E4b   E4b_meta vs R1: Δ = +0.0002 CI95 [-0.0052, +0.0056] p=0.9576 -> không chấp nhận


## 9. E5: phạm vi phủ định

Token nằm trong cửa sổ $k$ từ sau từ phủ định (`không`, `chẳng`, `chả`, `chưa`, `đừng`, kể cả từ ghép như `không_có`) được gắn tiền tố `not_`. Cửa sổ cố định vì `Clean Text` đã mất dấu câu. $k\in\{1,2,3\}$ được tune nested.

In [12]:
NEG = {"không", "chẳng", "chả", "chưa", "đừng"}

def neg_mark(toks, k):
    out, left = [], 0
    for w in toks:
        if w.split("_")[0] in NEG:
            out.append(w); left = k
        else:
            out.append(f"not_{w}" if left > 0 else w)
            left = max(left - 1, 0)
    return out

assert neg_mark(["phim", "không", "hay", "lắm", "đâu"], 2) == ["phim", "không", "not_hay", "not_lắm", "đâu"]
for k_ in (1, 2, 3):
    corpus(f"neg{k_}", lambda: [word_terms(neg_mark(t, k_)) for t in T_CLEAN])
run(NBVariant("E5_neg", corpora=("neg1", "neg2", "neg3")), "k tune nested")
compare("E5_neg", "R1", "E5")
print(pd.DataFrame([s for s in SELECTIONS if s["variant"] == "E5_neg"])["corpus"].value_counts().to_dict())
save_tables()

E5_neg         macro-F1 0.6102 ± 0.0060 | Neg 0.549 Neu 0.491 Pos 0.790 | acc 0.680 | 252 cấu hình, 2.6s
E5    E5_neg vs R1: Δ = -0.0026 CI95 [-0.0116, +0.0058] p=0.5368 -> không chấp nhận
{'neg2': 17, 'neg1': 8}


## 10. E6: đặc trưng ký tự và chọn đặc trưng

Char n-gram theo `char_wb`. Với char, `max_order` là độ dài n-gram lớn nhất. Hợp word + char gắn tiền tố `c|` để không trùng term.
Chọn đặc trưng dựa trên bảng 3×2 có/không có term, tính chỉ trên train của fold.

In [13]:
T_CHAR = [char_wb_terms(t) for t in dev["Clean Text"]]
corpus("char", lambda: T_CHAR)
corpus("union", lambda: [word_terms(a) + [("c|" + g, 2) for g, _ in b] for a, b in zip(T_CLEAN, T_CHAR)])
print("n_terms: char", CORPORA["char"].n_terms, "nnz", CORPORA["char"].doc.numel(), "| union", CORPORA["union"].n_terms, "nnz", CORPORA["union"].doc.numel())
run(NBVariant("E6a_char", corpora=("char",), max_orders=(4, 5), min_dfs=(2, 3, 5)), "char_wb 2 đến 4/5")
run(NBVariant("E6b_union", corpora=("union",), max_orders=(2,), min_dfs=(1, 2, 3)), "word(1,2) + char_wb(2,5)")
TOPKS = tuple((k_, m_) for m_ in ("chi2", "mi") for k_ in (500, 1000, 2000, 4000))
run(NBVariant("E6c_select", topks=TOPKS), "chi2/MI, K tune nested")
compare("E6a_char", "R1", "E6a")
compare("E6b_union", "R1", "E6b")
compare("E6c_select", "R1", "E6c")
save_tables()

n_terms: char 15685 nnz 321403 | union 41576 nnz 380904


E6a_char       macro-F1 0.5989 ± 0.0045 | Neg 0.537 Neu 0.489 Pos 0.770 | acc 0.663 | 84 cấu hình, 1.0s


E6b_union      macro-F1 0.6140 ± 0.0041 | Neg 0.549 Neu 0.499 Pos 0.794 | acc 0.687 | 42 cấu hình, 0.5s


E6c_select     macro-F1 0.6046 ± 0.0042 | Neg 0.541 Neu 0.484 Pos 0.789 | acc 0.676 | 672 cấu hình, 13.4s
E6a   E6a_char vs R1: Δ = -0.0140 CI95 [-0.0320, +0.0040] p=0.1244 -> không chấp nhận
E6b   E6b_union vs R1: Δ = +0.0012 CI95 [-0.0152, +0.0175] p=0.8824 -> không chấp nhận
E6c   E6c_select vs R1: Δ = -0.0082 CI95 [-0.0152, -0.0013] p=0.0200 -> không chấp nhận


## 11. E7: cân bằng lớp

- **E7a:** mỗi văn bản có trọng số $N/(C\,N_c)$ trong $\theta$ và prior.
- **E7b:** oversample lớp thiểu số bằng bội số ngẫu nhiên, chỉ trên train của fold. `df` và IDF vẫn tính trên dữ liệu gốc.
- **E7c:** cộng offset $\delta$ vào log-prior của Negative và Neutral, chọn ở inner CV.

In [14]:
def w_balanced(tr, yy):
    cc = torch.bincount(yy[tr], minlength=C).to(DTYPE)
    return (tr.sum().to(DTYPE) / (C * cc))[yy]

GEN_OS = torch.Generator(device=DEVICE).manual_seed(SEED)

def w_oversample(tr, yy):
    w = torch.ones(yy.numel(), dtype=DTYPE, device=DEVICE)
    cc = torch.bincount(yy[tr], minlength=C)
    top = cc.max()
    for c in range(C):
        idx = (tr & (yy == c)).nonzero().ravel()
        extra = int(top - cc[c])
        if extra:
            pick = idx[torch.randint(0, idx.numel(), (extra,), generator=GEN_OS, device=DEVICE)]
            w.index_add_(0, pick, torch.ones(extra, dtype=DTYPE, device=DEVICE))
    return w

run(NBVariant("E7a_balanced", weight_fn=w_balanced), "class weight balanced")
GEN_OS.manual_seed(SEED)
run(NBVariant("E7b_oversample", weight_fn=w_oversample), "bội số oversample trong train fold")
OFF = tuple(itertools.product((-1.0, -0.5, 0.0, 0.5, 1.0), repeat=2))
run(NBVariant("E7c_offset", offsets=OFF, priors=(False,)), "offset log-prior tune nested, prior nền đều")
compare("E7a_balanced", "R1", "E7a")
compare("E7b_oversample", "R1", "E7b")
compare("E7c_offset", "R1", "E7c")
save_tables()

E7a_balanced   macro-F1 0.6212 ± 0.0107 | Neg 0.565 Neu 0.511 Pos 0.787 | acc 0.680 | 84 cấu hình, 1.1s


E7b_oversample macro-F1 0.6152 ± 0.0064 | Neg 0.562 Neu 0.497 Pos 0.786 | acc 0.677 | 84 cấu hình, 1.2s


E7c_offset     macro-F1 0.6079 ± 0.0054 | Neg 0.545 Neu 0.492 Pos 0.786 | acc 0.678 | 1050 cấu hình, 1.0s
E7a   E7a_balanced vs R1: Δ = +0.0084 CI95 [-0.0020, +0.0181] p=0.1204 -> không chấp nhận
E7b   E7b_oversample vs R1: Δ = +0.0024 CI95 [-0.0072, +0.0117] p=0.6300 -> không chấp nhận
E7c   E7c_offset vs R1: Δ = -0.0049 CI95 [-0.0130, +0.0034] p=0.2456 -> không chấp nhận


## 12. Quyết định sau E2–E7 và phần thăm dò

Theo bản đăng ký: không biến thể nào trong E2–E7 có cận dưới CI95 > 0, nên không có gì để ghép và **NB tốt nhất là R1**.

Ô dưới đây là **thăm dò, không nằm trong bản đăng ký**. Nó trả lời câu hỏi: ghép hai biến thể có $\Delta$ điểm lớn nhất thì được bao nhiêu. Cặp này được chọn sau khi xem kết quả, nên điểm CV có thể bị thổi phồng. Sau khi thấy kết quả và trước khi mở test, người dùng quyết định đưa cấu hình này vào họ so sánh (Nhật ký sai lệch, mục 7 của `experiment_plan.md`).

In [15]:
accepted = [c for c in COMPARE if c["experiment"].startswith(("E2", "E3", "E4", "E5", "E6", "E7")) and c["accepted"]]
print("Biến thể E2–E7 được chấp nhận:", [c["variant"] for c in accepted])
assert not accepted, "có biến thể được chấp nhận: phải chạy bước ghép theo mục 4 của bản đăng ký"
BEST_NB, BEST_NB_VAR = "R1", R1
run(NBVariant("X_old_bal", corpora=("old",), weight_fn=w_balanced), "THĂM DÒ: E4a + E7a, chọn sau khi xem kết quả")
compare("X_old_bal", "R1", "X")
save_tables()

Biến thể E2–E7 được chấp nhận: []


X_old_bal      macro-F1 0.6391 ± 0.0047 | Neg 0.579 Neu 0.547 Pos 0.792 | acc 0.695 | 84 cấu hình, 1.1s
X     X_old_bal vs R1: Δ = +0.0263 CI95 [+0.0113, +0.0411] p=0.0004 -> CHẤP NHẬN


## 13. E8: kết hợp NB với LogReg

Mỗi outer fold dùng đúng cấu hình đã chọn cho fold đó (R1 và `LogReg_tuned`), tức không nhìn outer-test.

Xác suất của NB là softmax của joint log-likelihood. Với TF-IDF, đây là điểm số đã chuẩn hóa chứ không phải xác suất đã hiệu chuẩn.

- **E8a soft voting:** $p=w\,p_{\text{NB}}+(1-w)\,p_{\text{LR}}$, chọn $w\in\{0,0.25,0.5,0.75,1\}$ trên inner OOF.
- **E8b stacking:** LogReg meta ($C=1$, `class_weight="balanced"`) học trên log-xác suất inner OOF của hai mô hình.

In [16]:
from sklearn.metrics import f1_score

def sel_of(name):
    return {(s["repeat"], s["fold"]): s for s in SELECTIONS if s["variant"] == name}

def lr_proba_task(Xtr, ytr, Xev, c, cw):
    return make_model("logreg", c, cw).fit(Xtr, ytr).predict_proba(Xev)

def parse_cw(s):
    return None if s in (None, "None") else s

def ensembles(nb_name, nb_var, lr_name):
    t0 = time.perf_counter()
    s_nb, s_lr = sel_of(nb_name), sel_of(lr_name)
    cp = CORPORA["clean"]
    P_nb, tasks, keys = {}, [], []
    for r in range(R):
        for k in range(K):
            otr = Ft[r] != k
            c = s_lr[(r, k)]
            for q in range(Q):
                iv = otr & (It[r, k] == q)
                it = otr & ~iv
                P_nb[(r, k, q)] = torch.softmax(nb_var.predict(s_nb[(r, k)]["cfg"], it, iv), 1).cpu().numpy()
                X = csr_from(vectorize(cp, it, c["min_df"], c["max_order"], "tfidf"), N)
                a, b = it.cpu().numpy(), iv.cpu().numpy()
                tasks.append(delayed(lr_proba_task)(X[a], y_np[a], X[b], c["C"], parse_cw(c["class_weight"])))
                keys.append((r, k, q))
    P_lr = dict(zip(keys, Parallel(n_jobs=N_JOBS)(tasks)))
    WS = (0.0, 0.25, 0.5, 0.75, 1.0)
    pv, ps = np.zeros((R, N), dtype=np.int64), np.zeros((R, N), dtype=np.int64)
    pva = np.zeros((R, N, C))
    pa_all, pb_all = RESULTS[nb_name]["prob"].cpu().numpy(), RESULTS[lr_name]["prob"].cpu().numpy()
    log_ = lambda p: np.log(np.clip(p, 1e-12, 1.0))
    for r in range(R):
        for k in range(K):
            A = np.vstack([P_nb[(r, k, q)] for q in range(Q)])
            B = np.vstack([P_lr[(r, k, q)] for q in range(Q)])
            yy = np.concatenate([y_np[(F_np[r] != k) & (I_np[r, k] == q)] for q in range(Q)])
            fs = [f1_score(yy, (w * A + (1 - w) * B).argmax(1), average="macro") for w in WS]
            w = WS[int(np.argmax(fs))]
            ote = F_np[r] == k
            pa, pb = pa_all[r, ote], pb_all[r, ote]
            pva[r, ote] = w * pa + (1 - w) * pb
            pv[r, ote] = pva[r, ote].argmax(1)
            meta = LogisticRegression(C=1, class_weight="balanced", max_iter=2000).fit(np.hstack([log_(A), log_(B)]), yy)
            ps[r, ote] = meta.predict(np.hstack([log_(pa), log_(pb)]))
            SELECTIONS.append(dict(variant="E8a_softvote", repeat=r, fold=k, cfg=-1, inner_macro_f1=max(fs), w_nb=w, kind="softvote"))
    t = time.perf_counter() - t0
    for nm, p, note in (("E8a_softvote", pv, "NB + LogReg soft voting"), ("E8b_stacking", ps, "NB + LogReg stacking")):
        row = register(nm, torch.tensor(p, device=DEVICE), torch.tensor(pva, device=DEVICE) if nm == "E8a_softvote" else None, t, note)
        TABLE.append(row)
        print(f"{nm:14s} macro-F1 {row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f} | Neg {row['f1_negative']:.3f} "
              f"Neu {row['f1_neutral']:.3f} Pos {row['f1_positive']:.3f} | acc {row['accuracy']:.3f} | {t:.1f}s")

ensembles(BEST_NB, BEST_NB_VAR, "LogReg_tuned")
compare("E8a_softvote", BEST_NB, "E8a")
compare("E8b_stacking", BEST_NB, "E8b")
compare("E8a_softvote", "LogReg_tuned", "ref")
print(pd.DataFrame([s for s in SELECTIONS if s["variant"] == "E8a_softvote"])["w_nb"].value_counts().sort_index().to_dict())
save_tables()

E8a_softvote   macro-F1 0.6277 ± 0.0081 | Neg 0.567 Neu 0.515 Pos 0.801 | acc 0.694 | 1.8s
E8b_stacking   macro-F1 0.6157 ± 0.0067 | Neg 0.560 Neu 0.522 Pos 0.765 | acc 0.661 | 1.8s
E8a   E8a_softvote vs R1: Δ = +0.0149 CI95 [+0.0059, +0.0240] p=0.0020 -> CHẤP NHẬN
E8b   E8b_stacking vs R1: Δ = +0.0029 CI95 [-0.0096, +0.0159] p=0.6536 -> không chấp nhận
ref   E8a_softvote vs LogReg_tuned: Δ = +0.0083 CI95 [+0.0004, +0.0164] p=0.0396 -> CHẤP NHẬN
{0.0: 3, 0.25: 15, 0.5: 6, 0.75: 1}


## 14. E9: nhãn nghi nhiễu (nhánh riêng)

**Lọc trong CV:** trên mỗi outer-train, chạy OOF 3 fold với cấu hình R0, bỏ các dòng có dự đoán khác nhãn và độ tin cậy $>\tau=0.7$, rồi tune và fit trên phần còn lại. Điểm luôn tính trên **nhãn gốc**, nên phép đo có thể đánh giá thấp lợi ích nếu nhãn đánh giá cũng nhiễu.

**Tái tính 37/1491:** OOF 5 fold chỉ trên train với cấu hình R0. Quy trình gốc không có artifact, nên con số có thể lệch.

In [17]:
TAU = 0.7
FLAG_COUNTS = []

def flag_filter(r, k, otr):
    keep = torch.ones_like(otr)
    for q in range(Q):
        iv = otr & (It[r, k] == q)
        p = torch.softmax(R0.predict(0, otr & ~iv, iv), 1)
        conf, hat = p.max(1)
        bad = (hat != y[iv]) & (conf > TAU)
        keep[iv.nonzero().ravel()[bad]] = False
    FLAG_COUNTS.append(int((~keep).sum()))
    return keep

run(NBVariant("E9_filter"), "bỏ nhãn nghi nhiễu trong outer-train, chấm trên nhãn gốc", train_filter=flag_filter)
print("Số dòng bị bỏ mỗi outer fold: trung bình", np.mean(FLAG_COUNTS), "| min", min(FLAG_COUNTS), "| max", max(FLAG_COUNTS))
compare("E9_filter", "R1", "E9")
save_tables()

E9_filter      macro-F1 0.5980 ± 0.0096 | Neg 0.536 Neu 0.466 Pos 0.792 | acc 0.679 | 84 cấu hình, 1.1s
Số dòng bị bỏ mỗi outer fold: trung bình 220.4 | min 197 | max 251
E9    E9_filter vs R1: Δ = -0.0148 CI95 [-0.0251, -0.0046] p=0.0032 -> không chấp nhận


**E9b (POST-HOC, ngoài bản đăng ký):** E9 gắn cờ khoảng 15% outer-train vì NB với $\\alpha=0.01$ quá tự tin, nhiều hơn hẳn con số ~2.5% (37/1491) cần kiểm tra. Bản này chỉ bỏ dòng khi **cả NB (R0) và LogReg** ($C=3$, `balanced`) cùng đoán khác nhãn với độ tin cậy $>0.7$. Kết quả chỉ mang tính mô tả, không đưa vào test.

In [18]:
STRICT_COUNTS = []

def strict_filter(r, k, otr):
    keep = torch.ones_like(otr)
    for q in range(Q):
        iv = otr & (It[r, k] == q)
        it = otr & ~iv
        conf, hat = torch.softmax(R0.predict(0, it, iv), 1).max(1)
        X = csr_from(vectorize(CORPORA["clean"], it, 2, 2, "tfidf"), N)
        a, b = it.cpu().numpy(), iv.cpu().numpy()
        pl = torch.tensor(make_model("logreg", 3, "balanced").fit(X[a], y_np[a]).predict_proba(X[b]), device=DEVICE)
        cl, hl = pl.max(1)
        bad = (hat != y[iv]) & (conf > TAU) & (hl != y[iv]) & (cl > TAU)
        keep[iv.nonzero().ravel()[bad]] = False
    STRICT_COUNTS.append(int((~keep).sum()))
    return keep

run(NBVariant("E9b_strict"), "POST-HOC: bỏ dòng khi cả NB và LogReg đồng ý là nghi nhiễu", train_filter=strict_filter)
print("Số dòng bị bỏ mỗi outer fold: trung bình", np.mean(STRICT_COUNTS), "| min", min(STRICT_COUNTS), "| max", max(STRICT_COUNTS))
compare("E9b_strict", "R1", "E9b*")
save_tables()

E9b_strict     macro-F1 0.6079 ± 0.0108 | Neg 0.542 Neu 0.487 Pos 0.795 | acc 0.683 | 84 cấu hình, 3.6s
Số dòng bị bỏ mỗi outer fold: trung bình 40.36 | min 26 | max 57
E9b*  E9b_strict vs R1: Δ = -0.0049 CI95 [-0.0113, +0.0016] p=0.1316 -> không chấp nhận


In [19]:
# Danh sách ứng viên audit: chỉ trên train
tr_idx = np.where(dev["origin"].values == "train")[0]
oof_nb = np.zeros((N, C)); oof_lr = np.zeros((N, C))
sg = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
tasks, parts = [], []
for fit, ev in sg.split(tr_idx, y_np[tr_idx], gid_np[tr_idx]):
    a = torch.zeros(N, dtype=torch.bool, device=DEVICE); b = a.clone()
    a[torch.tensor(tr_idx[fit], device=DEVICE)] = True
    b[torch.tensor(tr_idx[ev], device=DEVICE)] = True
    oof_nb[b.cpu().numpy()] = torch.softmax(R0.predict(0, a, b), 1).cpu().numpy()
    X = csr_from(vectorize(CORPORA["clean"], a, 2, 2, "tfidf"), N)
    am, bm = a.cpu().numpy(), b.cpu().numpy()
    tasks.append(delayed(lr_proba_task)(X[am], y_np[am], X[bm], 3, "balanced")); parts.append(bm)
for bm, s in zip(parts, Parallel(n_jobs=N_JOBS)(tasks)):
    oof_lr[bm] = s

cand = pd.DataFrame({"Comment ID": dev["Comment ID"], "label": dev["Final Label"],
                     "nb_pred": [LABELS[i] for i in oof_nb.argmax(1)], "nb_conf": oof_nb.max(1),
                     "lr_pred": [LABELS[i] for i in oof_lr.argmax(1)], "lr_conf": oof_lr.max(1)}).iloc[tr_idx]
cand["flag_nb"] = (cand.nb_pred != cand.label) & (cand.nb_conf > TAU)
cand["flag_lr"] = (cand.lr_pred != cand.label) & (cand.lr_conf > TAU)
flagged = cand[cand.flag_nb | cand.flag_lr]
print("Train:", len(cand), "| cờ NB:", int(cand.flag_nb.sum()), "| cờ LogReg:", int(cand.flag_lr.sum()),
      "| cả hai:", int((cand.flag_nb & cand.flag_lr).sum()), "| hợp:", len(flagged))
flagged.to_csv(EXP / "label_audit_candidates.csv", index=False)

Train: 1491 | cờ NB: 214 | cờ LogReg: 61 | cả hai: 43 | hợp: 232


## 15. E10: augmentation đơn giản

Mỗi văn bản Negative hoặc Neutral có từ 3 token được thêm một bản sao: xóa ngẫu nhiên mỗi token với xác suất 0.1 rồi hoán đổi một cặp token. Bản sao chỉ tham gia khi văn bản gốc thuộc phần train của fold, và không tính vào `df`/IDF. Augmentation không thay thế dữ liệu thật.

In [20]:
rng_aug = np.random.default_rng(SEED)

def augment(toks):
    out = [w for w in toks if rng_aug.random() > 0.1] or list(toks)
    if len(out) > 1:
        i, j = rng_aug.choice(len(out), 2, replace=False)
        out[i], out[j] = out[j], out[i]
    return out

aug_idx = [i for i in range(N) if y_np[i] != L2I["Positive"] and len(T_CLEAN[i]) >= 3]
aug_toks = [augment(T_CLEAN[i]) for i in aug_idx]
corpus("aug", lambda: [word_terms(t) for t in T_CLEAN + aug_toks])
print("Số bản sao:", len(aug_idx))
run(NBVariant("E10_aug", corpora=("aug",), ext_src=torch.tensor(aug_idx, device=DEVICE)), "random deletion + swap cho lớp thiểu số")
compare("E10_aug", "R1", "E10")
save_tables()

Số bản sao: 680


E10_aug        macro-F1 0.6152 ± 0.0093 | Neg 0.557 Neu 0.492 Pos 0.796 | acc 0.688 | 84 cấu hình, 1.0s
E10   E10_aug vs R1: Δ = +0.0024 CI95 [-0.0055, +0.0102] p=0.5356 -> không chấp nhận


## 16. Độ nhạy theo $\alpha$

Bảng mô tả, không dùng để chọn tham số: cố định word (1,2), `min_df=2`, `fit_prior=False`, chỉ thay $\alpha$, chấm trên outer fold.
So với lưới val cũ để thấy lựa chọn $\alpha=0.01$ phụ thuộc vào 309 mẫu val đến mức nào.

In [21]:
av = NBVariant("alpha_scan", max_orders=(2,), min_dfs=(2,), priors=(False,))
cms = torch.zeros(R, len(ALPHAS), C, C, dtype=DTYPE, device=DEVICE)
for r in range(R):
    for k in range(K):
        cms[r] += av.score(Ft[r] != k, Ft[r] == k)
f = score_macro(cms).cpu().numpy()
g_val = pd.read_csv(NBDIR / "grid_search_val.csv")
g_val = g_val[(g_val.ngram == "(1, 2)") & (g_val.min_df == 2) & (~g_val.fit_prior)].set_index("alpha")["val_macro_f1"]
alpha_tbl = pd.DataFrame({"alpha": ALPHAS, "cv_macro_f1_mean": f.mean(0), "cv_macro_f1_std": f.std(0, ddof=1),
                          "val_macro_f1_old_grid": [g_val.get(a, np.nan) for a in ALPHAS]})
alpha_tbl.to_csv(EXP / "alpha_sensitivity.csv", index=False)
print(alpha_tbl.round(4).to_string(index=False))

 alpha  cv_macro_f1_mean  cv_macro_f1_std  val_macro_f1_old_grid
 0.001            0.5709           0.0059                    NaN
 0.003            0.5784           0.0033                    NaN
 0.010            0.5928           0.0046                 0.6147
 0.030            0.6057           0.0071                 0.5936
 0.100            0.6122           0.0031                 0.5799
 0.300            0.6095           0.0077                 0.5845
 1.000            0.5252           0.0072                 0.4843


## 17. Learning curve

Cấu hình cố định theo cấu hình R1 được chọn nhiều nhất. Mỗi outer-train được lấy mẫu con theo tỉ lệ, 3 seed lấy mẫu mỗi mức, chấm trên outer-test.

- **Toàn bộ:** mọi lớp cùng tỉ lệ.
- **Theo lớp:** chỉ giảm một lớp, giữ nguyên hai lớp còn lại. Đường này cho biết thêm mẫu lớp nào có lợi nhất.

Ngoại suy bằng $F(n)=a+b\ln n$ trên 4 điểm. Đây là **ước lượng thô**, không phải cam kết.

In [22]:
sel = pd.DataFrame([s for s in SELECTIONS if s["variant"] == "R1"])
mode = sel[["alpha", "min_df", "max_order", "fit_prior"]].value_counts().index[0]
LC_ALPHA, LC_MD, LC_MO, LC_FP = float(mode[0]), int(mode[1]), int(mode[2]), bool(mode[3])
print("Cấu hình learning curve:", dict(alpha=LC_ALPHA, min_df=LC_MD, max_order=LC_MO, fit_prior=LC_FP))
lcv = NBVariant("LC", max_orders=(LC_MO,), min_dfs=(LC_MD,), alphas=(LC_ALPHA,), priors=(LC_FP,))
FRACS = (0.25, 0.5, 0.75, 1.0)

def subsample(otr_np, frac, rng, only=None):
    m = np.zeros(N, dtype=bool)
    for c in range(C):
        idx = np.where(otr_np & (y_np == c))[0]
        f = frac if only in (None, c) else 1.0
        m[rng.choice(idx, int(round(f * len(idx))), replace=False)] = True
    return m

rows = []
for curve, only in (("all", None), ("Negative", 0), ("Neutral", 1), ("Positive", 2)):
    for frac in FRACS:
        per, ns, nc = [], [], []
        for s in range(3):
            rng = np.random.default_rng(1000 * s + int(frac * 100) + (0 if only is None else only + 1))
            pred = torch.zeros(R, N, dtype=torch.long, device=DEVICE)
            for r in range(R):
                for k in range(K):
                    otr_np = F_np[r] != k
                    m = subsample(otr_np, frac, rng, only)
                    ote = Ft[r] == k
                    pred[r, ote] = lcv.predict(0, torch.tensor(m, device=DEVICE), ote).argmax(1)
                    ns.append(m.sum())
                    nc.append(np.bincount(y_np[m], minlength=C))
                cm = cm_batch(y, pred[r][None])
                per.append((float(score_macro(cm)), *macro_f1_cm(cm)[0].tolist()))
        per = np.array(per)
        nc_ = np.mean(nc, 0)
        rows.append(dict(curve=curve, frac=frac, n_train=float(np.mean(ns)), n_negative=nc_[0], n_neutral=nc_[1], n_positive=nc_[2],
                         macro_f1_mean=per[:, 0].mean(), macro_f1_std=per[:, 0].std(ddof=1),
                         f1_negative=per[:, 1].mean(), f1_neutral=per[:, 2].mean(), f1_positive=per[:, 3].mean(), n_runs=len(per)))
lc = pd.DataFrame(rows)
lc.to_csv(EXP / "learning_curve.csv", index=False)
print(lc.round(4).drop(columns=["n_runs"]).to_string(index=False))

ex = []
for curve in ("all", "Negative", "Neutral", "Positive"):
    d = lc[lc.curve == curve]
    x = d["n_train"].values if curve == "all" else d[f"n_{curve.lower()}"].values
    b, a = np.polyfit(np.log(x), d["macro_f1_mean"].values, 1)
    cur = x[-1]
    for add in (100, 200, 400, 800):
        ex.append(dict(curve=curve, n_current=cur, n_added=add, b=b, macro_f1_current=a + b * np.log(cur),
                       macro_f1_pred=a + b * np.log(cur + add), gain_pred=b * np.log((cur + add) / cur)))
ext = pd.DataFrame(ex)
ext.to_csv(EXP / "learning_curve_extrapolation.csv", index=False)
print(ext.round(4).to_string(index=False))

Cấu hình learning curve: {'alpha': 0.1, 'min_df': 2, 'max_order': 2, 'fit_prior': False}


   curve  frac  n_train  n_negative  n_neutral  n_positive  macro_f1_mean  macro_f1_std  f1_negative  f1_neutral  f1_positive
     all  0.25   360.20       63.48      78.48      218.24         0.5425        0.0109       0.4543      0.4340       0.7393
     all  0.50   720.20      126.84     156.92      436.44         0.5767        0.0095       0.5043      0.4610       0.7646
     all  0.75  1079.80      190.12     235.12      654.56         0.6000        0.0069       0.5338      0.4821       0.7841
     all  1.00  1440.00      253.60     313.60      872.80         0.6122        0.0029       0.5515      0.4928       0.7922
Negative  0.25  1249.88       63.48     313.60      872.80         0.5443        0.0086       0.3548      0.4969       0.7811
Negative  0.50  1313.24      126.84     313.60      872.80         0.5946        0.0063       0.4961      0.4965       0.7913
Negative  0.75  1376.52      190.12     313.60      872.80         0.6061        0.0060       0.5341      0.4924      

## 18. Mẫu phân tích lỗi

Lấy 5 mẫu sai cho mỗi cặp nhầm lẫn từ OOF của R1 (repeat 0), tổng 30 mẫu, seed cố định. Phần này chỉ dùng dev. File tạm có `Clean Text` chưa che tên nên được ghi ra ngoài repo; bản phân loại cuối đã che tên nằm ở `experiments/error_analysis_labeled.csv`.

In [23]:
pred0 = RESULTS["R1"]["pred"][0].cpu().numpy()
conf0 = RESULTS["R1"]["prob"][0].max(1).values.cpu().numpy()
cm_dev = np.zeros((C, C), dtype=int)
np.add.at(cm_dev, (y_np, pred0), 1)
pd.DataFrame(cm_dev, index=LABELS, columns=LABELS).to_csv(EXP / "confusion_R1_oof_r0.csv")
print(pd.DataFrame(cm_dev, index=LABELS, columns=LABELS))
pick = []
rng = np.random.default_rng(SEED)
for a, b in itertools.permutations(range(C), 2):
    idx = np.where((y_np == a) & (pred0 == b))[0]
    uniq = dev.iloc[idx].drop_duplicates("Clean Text").index.values
    pick += list(rng.choice(uniq, 5, replace=False))
err = dev.iloc[pick][["Comment ID", "Clean Text", "Final Label"]].copy()
err["pred"] = [LABELS[i] for i in pred0[pick]]
err["conf"] = conf0[pick].round(3)
import tempfile
ERR_TMP = Path(os.environ.get("ERR_SAMPLE_PATH", Path(tempfile.gettempdir()) / "error_sample_unmasked.csv"))
err.to_csv(ERR_TMP, index=False)
print("Đã lấy", len(err), "mẫu")

          Negative  Neutral  Positive
Negative       179       47        91
Neutral         74      183       135
Positive       104      124       863
Đã lấy 30 mẫu


## 19. Khóa ứng viên cho test

Bản nested CV ước lượng hiệu năng của *quy trình* và có thể chọn cấu hình khác nhau ở mỗi fold. Để có một cấu hình duy nhất cho lần fit cuối trên train+val, mỗi họ mô hình chọn cấu hình tốt nhất theo macro-F1 gộp trên 25 outer fold.

Mô hình lai là mô hình có điểm nested cao nhất trong E3b, E3b*, E8a, E8b.

In [24]:
def select_on_dev(v):
    cms = 0.0
    for r in range(R):
        for k in range(K):
            cms = cms + v.score(Ft[r] != k, Ft[r] == k)
    f = score_macro(cms)
    b = first_best(f)
    return v.configs[b], float(f[b])

nb_cfg, nb_score = select_on_dev(R1)

def lr_cv_task(Xtr, ytr, Xev, yev, grid):
    return sk_grid_task(Xtr, ytr, Xev, yev, "logreg", grid)

grid = list(itertools.product(LR_C, (None, "balanced")))
blocks = list(itertools.product((1, 2), (1, 2, 3)))
tasks, keys = [], []
for r in range(R):
    for k in range(K):
        otr = Ft[r] != k
        for bi, (mo, md_) in enumerate(blocks):
            X = csr_from(vectorize(CORPORA["clean"], otr, md_, mo, "tfidf"), N)
            a = otr.cpu().numpy()
            tasks.append(delayed(lr_cv_task)(X[a], y_np[a], X[~a], y_np[~a], grid)); keys.append(bi)
agg = {}
for bi, cm in zip(keys, Parallel(n_jobs=N_JOBS)(tasks)):
    agg[bi] = agg.get(bi, 0) + cm
f = torch.cat([score_macro(torch.tensor(agg[bi], dtype=DTYPE, device=DEVICE).view(-1, C, C)) for bi in range(len(blocks))])
b = first_best(f)
bi, gi = divmod(b, len(grid))
lr_cfg = dict(max_order=blocks[bi][0], min_df=blocks[bi][1], C=grid[gi][0], class_weight=grid[gi][1])

res = pd.DataFrame(TABLE).set_index("variant")
hyb = res.loc[["E3b_nblr", "E3b_cw", "E8a_softvote", "E8b_stacking"], "macro_f1_mean"]
hyb_name = hyb.idxmax()
print("Mô hình lai:", hyb.round(4).to_dict(), "->", hyb_name)

hybrid = {"name": hyb_name}
if hyb_name == "E8a_softvote":
    P, Q_ = RESULTS["R1"]["prob"], RESULTS["LogReg_tuned"]["prob"]
    WS = (0.0, 0.25, 0.5, 0.75, 1.0)
    fs = []
    for w in WS:
        p = (w * P + (1 - w) * Q_).argmax(2)
        fs.append(float(np.mean([float(score_macro(cm_batch(y, p[r][None]))) for r in range(R)])))
    hybrid.update(method="soft_vote", w_nb=WS[int(np.argmax(fs))], grid_scores=dict(zip(map(str, WS), fs)))
elif hyb_name == "E8b_stacking":
    hybrid.update(method="stacking")
else:
    s = pd.DataFrame([x for x in SELECTIONS if x["variant"] == hyb_name])
    m = s[["C", "class_weight", "min_df"]].value_counts().index[0]
    hybrid.update(method="nblr", C=float(m[0]), class_weight=parse_cw(m[1]), min_df=int(m[2]))

x_var = NBVariant("X_cfg", corpora=("old",), weight_fn=w_balanced)
x_cfg, x_score = select_on_dev(x_var)

final = {"locked_at": time.strftime("%Y-%m-%d %H:%M:%S %Z"), "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
         "R0": dict(corpus="clean", weighting="tfidf", max_order=2, min_df=2, alpha=0.01, fit_prior=False),
         "NB_best": {"variant": "R1", **{k_: nb_cfg[k_] for k_ in ("corpus", "weighting", "max_order", "min_df", "alpha", "fit_prior")}, "cv_score_on_outer_folds": nb_score},
         "LogReg": {**lr_cfg, "cv_score_on_outer_folds": float(f[b])},
         "Hybrid": hybrid,
         "Exploratory_X_old_bal": {"variant": "X_old_bal", "eligible_for_test": True, "weights": "balanced",
                                   **{k_: x_cfg[k_] for k_ in ("corpus", "weighting", "max_order", "min_df", "alpha", "fit_prior")},
                                   "cv_score_on_outer_folds": x_score}}
(EXP / "final_candidates.json").write_text(json.dumps(final, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(final, ensure_ascii=False, indent=2))

Mô hình lai: {'E3b_nblr': 0.564, 'E3b_cw': 0.5966, 'E8a_softvote': 0.6277, 'E8b_stacking': 0.6157} -> E8a_softvote


{
  "locked_at": "2026-11-05 21:52:04 EST",
  "git_commit": "7a78a4c0152b9581a6296e77c426676d64e4ad86",
  "R0": {
    "corpus": "clean",
    "weighting": "tfidf",
    "max_order": 2,
    "min_df": 2,
    "alpha": 0.01,
    "fit_prior": false
  },
  "NB_best": {
    "variant": "R1",
    "corpus": "clean",
    "weighting": "tfidf",
    "max_order": 2,
    "min_df": 3,
    "alpha": 0.1,
    "fit_prior": false,
    "cv_score_on_outer_folds": 0.6134911824909162
  },
  "LogReg": {
    "max_order": 2,
    "min_df": 1,
    "C": 1,
    "class_weight": "balanced",
    "cv_score_on_outer_folds": 0.6231286330710457
  },
  "Hybrid": {
    "name": "E8a_softvote",
    "method": "soft_vote",
    "w_nb": 0.5,
    "grid_scores": {
      "0.0": 0.6193477941720025,
      "0.25": 0.6265614966276795,
      "0.5": 0.6287797209371766,
      "0.75": 0.6223924965271534,
      "1.0": 0.612802394656591
    }
  },
  "Exploratory_X_old_bal": {
    "variant": "X_old_bal",
    "eligible_for_test": true,
    "weights"

In [25]:
info = {"seed": SEED, "repeats": R, "outer_folds": K, "inner_folds": Q, "bootstrap_B": int(W_BOOT.shape[0]),
        "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        "python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
        "sklearn": sklearn.__version__, "pandas": pd.__version__, "gpu": torch.cuda.get_device_name(0), "n_cpu_affinity": N_JOBS,
        "omp_num_threads": os.environ.get("OMP_NUM_THREADS"), "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
        "hostname": platform.node(), "n_dev": N, "n_groups_dev": int(gid.max()) + 1}
(EXP / "run_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
save_tables()
print(pd.DataFrame(COMPARE)[["experiment", "variant", "reference", "delta", "ci95_lo", "ci95_hi", "p_boot", "accepted"]].round(4).to_string(index=False))

experiment        variant    reference   delta  ci95_lo  ci95_hi  p_boot  accepted
        E1             R1           R0  0.0200   0.0079   0.0327  0.0016      True
       ref   LogReg_tuned           R0  0.0265   0.0087   0.0451  0.0048      True
       ref   LogReg_tuned           R1  0.0065  -0.0071   0.0203  0.3400     False
       ref      SVC_tuned           R1 -0.0120  -0.0265   0.0024  0.1064     False
       ref             R0     Majority  0.3412   0.3169   0.3645  0.0002      True
       E2a      E2a_count           R1 -0.0089  -0.0211   0.0028  0.1368     False
       E2b     E2b_binary           R1 -0.0055  -0.0165   0.0052  0.3144     False
       E3a        E3a_cnb           R1 -0.0137  -0.0235  -0.0044  0.0032     False
       E3b       E3b_nblr           R1 -0.0488  -0.0669  -0.0309  0.0002     False
      E3b*         E3b_cw           R1 -0.0162  -0.0321  -0.0004  0.0464     False
       E4a        E4a_old           R1  0.0073  -0.0075   0.0219  0.3272     False
    